In [13]:
import numpy as np
import pandas as pd
from scipy.special import erfi

# np.random.seed(42)


In [14]:
N = 1_000_000
u1 = np.random.uniform(0, 1, N)
u2 = np.random.uniform(0, 1, N)


def mc(f,n=1):
    return (f(u1)).mean() if n == 1 else (f(u1,u2)).mean()


6.  $$
    \begin{aligned}
     &\int_{-2}^2 e^{x+x^2} \\
    =& 4\int_0^1 \exp\{4y-2 + (4y-2)^2 \} dy \quad y=\frac{x+2}{4} \iff x=4y-2 \\
    =& 4\int_0^1 \exp\{16y^2-12y+2 \} dy
    \end{aligned}
    $$


In [15]:
mc(lambda x: 4 * np.exp(16 * x**2 - 12 * x + 2))

np.float64(92.80245797825175)

$$
erfi(x)=\frac{2}{\sqrt{\pi}}\int_0^x e^{t^2} dt
$$

$$
\begin{aligned}
\int_{-2}^2 e^{x+x^2} dx &= e^{-1/4} \int_{-2}^2 e^{(x+1/2)^2} dx \\
& = e^{-1/4} \int_{-1.5}^{2.5} e^{t^2} dt \quad t = x+\frac{1}{2} \\
& = e^{-1/4} \frac{\sqrt{\pi}}{2} (erfi(2.5)-erfi(-1.5))
\end{aligned}
$$


In [16]:
np.exp(-0.25) * np.pi**0.5 / 2 * (erfi(2.5) - erfi(-1.5))

np.float64(93.16275329244198)

12.


In [17]:
def simulate_N():
    s = 0
    n = 0
    while s <= 1:
        s += np.random.uniform(0, 1)
        n += 1
    return n


def estimate_EN(num_sim):
    results = np.array([simulate_N() for _ in range(num_sim)])
    return np.mean(results)


print("100 sim:", estimate_EN(100))
print("1000 sim:", estimate_EN(1000))
print("10000 sim:", estimate_EN(10000))

100 sim: 2.63
1000 sim: 2.683
10000 sim: 2.7269


$E[N]\to e$


6

$$
\begin{aligned}
&\int_0^\infty \frac{x}{(1+x^2)^2}dx \\
=&\int_0^1 \frac{1}{2} dy,\quad y=\frac{1}{1+x^2},\quad \frac{dy}{dx}=-2x(1+x^2)^{-2} \\
=&\frac{1}{2}
\end{aligned}
$$


In [18]:
mc(lambda x: np.ones_like(x) * 0.5)

np.float64(0.5)

13


In [19]:
def simulate_N_product():
    product = 1.0
    n = 0
    th = np.exp(-3)
    while product >= th:
        product *= np.random.uniform(0, 1)
        n += 1
    return n - 1


def run_simulation(num_sim):
    samples = np.array([simulate_N_product() for _ in range(num_sim)])
    x, counts = np.unique(samples, return_counts=True)
    prob = counts / num_sim

    return samples.mean(), pd.DataFrame({"N": x, "P": prob})


EN, prob = run_simulation(1000)

print("Estimated E[N]:", EN)
prob

Estimated E[N]: 3.031


,N,P
0,0,0.043
1,1,0.170
2,2,0.221
3,3,0.199
4,4,0.171
5,5,0.098
6,6,0.051
7,7,0.036
8,8,0.009
9,9,0.002


7.  $$
    \begin{aligned}
    \int_{-\infty}^\infty e^{-x^2} dx &= 2\int_0^\infty e^{-x^2} dx \quad y=\frac{1}{1+x}\iff x=\frac{1}{y}-1 \\
    &=2\int_0^1 \frac{1}{y^2}e^{-(1/y-1)^2} dy \quad \frac{dy}{dx}=-(1+x)^{-2}=-y^2
    \end{aligned}
    $$


In [20]:
mc(lambda x: 2 / x**2 * np.exp(-((1 / x - 1) ** 2))), np.sqrt(np.pi)

(np.float64(1.771726923005158), np.float64(1.7724538509055159))

8.


In [21]:
v = np.sqrt(1 - u1**2)

print(f"Corr(U, sqrt(1-U^2)): {np.corrcoef(u1, v)[0, 1]}")
print(f"Corr(U^2, sqrt(1-U^2)): {np.corrcoef(u1**2, v)[0, 1]}")

Corr(U, sqrt(1-U^2)): -0.9213325122912366
Corr(U^2, sqrt(1-U^2)): -0.9835545857883883


8.  $$
    \begin{aligned}
    \int_0^1\int_0^1 e^{(x+y)^2} dydx &= \int_0^1\int_x^{x+1} e^{u^2} dudx \quad u=x+y \\
    &= \int_0^1 \int_0^u e^{u^2} dxdu + \int_1^2\int_{u-1}^1 e^{u^2} dxdu \\
    &=\int_0^1 ue^{u^2} du + \int_1^2(2-u)e^{u^2} du \\
    &=\frac{3}{2}(e-1)-\frac{1}{2}e^4+\sqrt{\pi}[erfi(2)-erfi(1)]
    \end{aligned}
    $$


In [22]:
(
    mc(lambda x, y: np.exp((x + y) ** 2), 2),
    3 / 2 * (np.e - 1) - np.e**4 / 2 + np.pi**0.5 * (erfi(2) - erfi(1)),
)

(np.float64(4.903051305393981), np.float64(5.258299765316551))

In [23]:
def rand(x1: int, x2: int, n: int) -> list[int]:
    sep = [x1, x2] + [0] * (n - 2)
    for i in range(2, n):
        sep[i] = (3 * sep[i - 1] + 5 * sep[i - 2]) % 100

    return sep


print([i / 100 for i in rand(23, 66, 14)])

[0.23, 0.66, 0.13, 0.69, 0.72, 0.61, 0.43, 0.34, 0.17, 0.21, 0.48, 0.49, 0.87, 0.06]


8.  $$
    \int_0^\infty \int_0^x e^{-(x+y)} dydx = \int_0^\infty e^{-x} \int_0^x e^{-y} dydx = E[I(Y\le X)],\quad X,Y \overset{iid}{\sim} \exp(1)
    $$
    $$
    U\sim(0,1) \implies -\ln(U) \sim \exp(1)
    $$


In [29]:
e1, e2 = -np.log(u1), -np.log(u2)
np.sum(e1 < e2) / N

np.float64(0.49912)

$$
\int_0^\infty \int_0^x e^{-(x+y)} dydx = \int_0^\infty e^{-x}(1-e^{-x}) dx = \frac{1}{2}
$$


10. $$
    E[e^U] = \int_0^1 e^u du = e-1 ,\quad E[Ue^U] = \int_0^1 ue^u du = ue^u|^1_0 - \int_0^1 e^u du = 1
    $$
    $$
    Cov(U,e^U)=E[Ue^U] - E[U]E[e^U] = 1 - \frac{1}{2}(e-1) = \frac{3-e}{2}
    $$


In [33]:
(np.cov(u1, np.exp(u1))[0, 1], (3 - np.e) / 2)

(np.float64(0.14082793707107386), 0.14085908577047745)